# ID6004W Industrial Vision - Assignment 1 Solution Key
### Lectures 01-02: Image Filtering and Convolution

| | |
|---|---|
| **Released** | Oct 1st 2026 |
| **Due** | Oct 25th 2026 (5.00 PM IST) |
| **Total** | 100 marks (will be scaled down appropriately later) |

> **Instructor copy - do not distribute.** Every *Solution* cell below replaces the student's answer cell; the check cells recompute the numbers of the key and `assert` them, so *Run all* verifies the key.

## Instructions

**Working in Colab.** Open this notebook in Google Colab and immediately use *File → Save a copy in Drive*; work only in your copy. Type your answers in the cells marked **Your answer**, directly below each question (double-click a cell to edit it). Mathematics can be typed in LaTeX between `$...$` or `$$...$$`. Photographs of hand-written working are **not** accepted in place of typed answers.

- **The writing must be your own.**
- Cells marked **Optional check** are not graded. Use them to verify a hand calculation numerically, but the answer you submit must be derived by hand with the steps shown.
-  **Submission.** When you are done, use *File → Download → Download .ipynb*, rename the file to `<ROLLNO>_assignment1.ipynb`, and upload it here: [SUBMISSION_LINK](https://forms.gle/pRwExrMB1dvKoU138). For queries contact Rishabh Shukla (da26e011@smail.iitm.ac.in). 
- **Late policy:** 10 marks will be deducted for each day the assignment is submitted late after the due date.

## Student details

- **Name:**
- **Roll number:**

In [2]:
# Imports used by the optional check cells.
import numpy as np
from scipy import ndimage
rng = np.random.default_rng(0)

---
## Question 1. What is, and what is not, a convolution  [25 marks]

In Lecture 01 we proved that linearity together with shift invariance forces an operator to be a convolution. This question asks you to use that theorem as a *test*.

**(a)** For each operator $\mathcal{T}$ below, state whether it is (i) linear and (ii) shift invariant. If it is both, give its impulse response $h$ explicitly (as a small array, or as a formula). Here $f$ is the input image and $g=\mathcal{T}\{f\}$. **[15]**

1. $g(m,n) = f(m,n) - f(m-1,n)$
2. $g(m,n) = \operatorname{median}\big\{f(m+i,n+j) : |i|\le 1, |j|\le 1\big\}$
3. $g(m,n) = c(m,n)\,f(m,n)$, where $c$ is a fixed, non-constant mask (e.g. lens vignetting)
4. $g(m,n) = f(m,n)^{2}$
5. $g(m,n) = f(n,m)$ (transpose the image)
6. $g(m,n) = f(m,n) + 10$ (brighten)

**Solution 1(a).**

We use $g(m,n)=\sum_{m',n'}h(m',n')\,f(m-m',n-n')$.

1. Linear **and** shift invariant (LSI). $h(0,0)=1$, $h(1,0)=-1$, all other taps $0$.
2. Shift invariant, **not linear**. Example on $1\times3$ windows: $f_1=(0,0,1)$ and $f_2=(1,0,0)$ both have median $0$, but $f_1+f_2=(1,0,1)$ has median $1\neq0+0$.
3. Linear, **not shift invariant**. The weight $c(m,n)$ changes from pixel to pixel.
4. Shift invariant, **not linear**: $(2f)^2=4f^2\neq2f^2$.
5. Linear, **not shift invariant**: a shift down in the input becomes a shift right in the output.
6. Shift invariant, **not linear**: $\mathcal{T}\{2f\}=2f+10$, but $2\mathcal{T}\{f\}=2f+20$. (It is affine, not linear.)

Only operator 1 is LSI, so only operator 1 is a convolution.

In [3]:
# Check for 1(a): numerical tests of the two properties, with periodic boundaries so that shifts are exact.
N = 16
c_mask = rng.uniform(0.5, 1.0, (N, N))          # a fixed, non-constant mask

T = {
    1: lambda f: f - np.roll(f, 1, axis=0),
    2: lambda f: ndimage.median_filter(f, size=3, mode="wrap"),
    3: lambda f: c_mask * f,
    4: lambda f: f ** 2,
    5: lambda f: f.T,
    6: lambda f: f + 10,
}

def shift(f, dm, dn):
    return np.roll(f, (dm, dn), axis=(0, 1))

def is_linear(op, trials=5):
    for _ in range(trials):
        f1, f2 = rng.normal(size=(N, N)), rng.normal(size=(N, N))
        a, b = rng.normal(size=2)
        if not np.allclose(op(a * f1 + b * f2), a * op(f1) + b * op(f2)):
            return False
    return True

def is_shift_invariant(op):
    f = rng.normal(size=(N, N))
    return all(np.allclose(op(shift(f, dm, dn)), shift(op(f), dm, dn)) for dm, dn in ((1, 2), (3, 0), (0, 5)))

res = {k: (is_linear(op), is_shift_invariant(op)) for k, op in T.items()}
for k, (lin, si) in res.items():
    print(f"operator {k}: linear={lin}, shift invariant={si}")
assert res == {1: (True, True), 2: (False, True), 3: (True, False), 4: (False, True), 5: (True, False), 6: (False, True)}

delta = np.zeros((N, N)); delta[0, 0] = 1
h = T[1](delta)
print("impulse response of operator 1, non-zero taps:", {tuple(map(int, i)): h[tuple(i)] for i in np.argwhere(h)})
assert h[0, 0] == 1 and h[1, 0] == -1 and np.count_nonzero(h) == 2

operator 1: linear=True, shift invariant=True
operator 2: linear=False, shift invariant=True
operator 3: linear=True, shift invariant=False
operator 4: linear=False, shift invariant=True
operator 5: linear=True, shift invariant=False
operator 6: linear=False, shift invariant=True
impulse response of operator 1, non-zero taps: {(0, 0): np.float64(1.0), (1, 0): np.float64(-1.0)}


**(b)** Let $\mathcal{T}_1$ and $\mathcal{T}_2$ be two LSI systems with impulse responses $h_1$ and $h_2$. Show that the cascade $\mathcal{T}_2\{\mathcal{T}_1\{f\}\}$ is again LSI, identify its impulse response, and state why the order of the two systems does not matter. **[10]**

**Solution 1(b).**

If each step is linear, the cascade is linear. If each step commutes with shifts, so does the cascade. So the cascade is LSI, and therefore a convolution. To find its kernel, feed in $\delta$: $\mathcal{T}_1$ gives $h_1$, then $\mathcal{T}_2$ gives $h_1\ast h_2$. So the impulse response is $h_1\ast h_2$. Convolution is commutative, $h_1\ast h_2=h_2\ast h_1$ (substitute $m''=m-m'$ in the sum), so the order does not matter.

---
## Question 2. Separability and rank  [25 marks]

**(a)** Determine the rank of each kernel below, and for those of rank one give an explicit factorisation $H=h_1h_2^{\top}$. Do this by hand, using the fact that a nonzero matrix has rank one exactly when every $2\times2$ minor vanishes. **[15]**

$$
A=\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
B=\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}
\quad
C=\begin{bmatrix}1&2&3\\2&4&6\\3&6&9\end{bmatrix}
\quad
D=\begin{bmatrix}1&1&1\\1&2&1\\1&1&1\end{bmatrix}
$$

**Solution 2(a).**

- $A$: every row is a multiple of $[1,2,1]$, so all $2\times2$ minors are $0$. Rank 1: $A=\tfrac14[1,2,1]^{\top}\cdot\tfrac14[1,2,1]$.
- $C$: every row is a multiple of $[1,2,3]$, so all $2\times2$ minors are $0$. Rank 1: $C=[1,2,3]^{\top}[1,2,3]$.
- $B$: rows 1 and 3 are equal, and the minor $\begin{vmatrix}0&1\\1&-4\end{vmatrix}=-1\neq0$. Rank 2, not separable.
- $D$: rows 1 and 3 are equal, and the minor $\begin{vmatrix}1&1\\1&2\end{vmatrix}=1\neq0$. Rank 2, not separable.

Note: $A$ and $D$ are both smoothing kernels with a heavier centre, but only $A$ factors.

In [4]:
# Check for 2(a).
A = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16
B = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
C = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])
D = np.array([[1, 1, 1], [1, 2, 1], [1, 1, 1]])

ranks = [np.linalg.matrix_rank(K) for K in (A, B, C, D)]
print("ranks of A, B, C, D:", ranks)
assert ranks == [1, 2, 1, 2]
assert np.allclose(np.outer([1, 2, 1], [1, 2, 1]) / 16, A)
assert np.allclose(np.outer([1, 2, 3], [1, 2, 3]), C)

M = 2 * int(np.ceil(3 * 5)) + 1
print(f"2(b): M = {M}, direct {M * M}, separable {2 * M}, speed-up {M * M / (2 * M)}")
assert (M, M * M, 2 * M, M / 2) == (31, 961, 62, 15.5)

ranks of A, B, C, D: [np.int64(1), np.int64(2), np.int64(1), np.int64(2)]
2(b): M = 31, direct 961, separable 62, speed-up 15.5


**(b)** An $N\times N$ image is smoothed with a Gaussian of $\sigma=5$, truncated to $2\lceil 3\sigma\rceil+1$ taps. State the kernel size $M$, the number of multiplications per pixel for a direct 2D pass and for two 1D passes, and the resulting speed-up. **[10]**

**Solution 2(b).**

$M=2\lceil3\cdot5\rceil+1=31$. Direct 2D pass: $M^{2}=961$ multiplications per pixel. Two 1D passes: $2M=62$. Speed-up: $961/62=M/2=15.5\times$.

---
## Question 3. Noise gain  [25 marks]

Let the noise be i.i.d. with variance $\sigma_\eta^{2}$, so that a linear filter $w$ produces output noise variance $\sigma_\eta^{2}\sum_k w_k^{2}$.

**(a)** Compute $\sum_k w_k^{2}$ for each of the four kernels below, and give the output noise standard deviation as a multiple of $\sigma_\eta$. **[10]**

$$
\text{box } \tfrac19\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}
\quad
\text{Gaussian } \tfrac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
\text{Sobel } \begin{bmatrix}1&0&-1\\2&0&-2\\1&0&-1\end{bmatrix}
\quad
\text{sharpen } \begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}
$$

**Solution 3(a).**

| kernel | $\sum_k w_k^2$ | output std |
|---|---|---|
| box | $9\cdot(1/9)^2=1/9\approx0.111$ | $0.333\,\sigma_\eta$ |
| Gaussian | $(1+4+1+4+16+4+1+4+1)/256=36/256=0.1406$ | $0.375\,\sigma_\eta$ |
| Sobel | $1+4+1+1+4+1=12$ | $3.46\,\sigma_\eta$ |
| sharpen | $25+4\cdot1=29$ | $5.39\,\sigma_\eta$ |

Sharpening boosts noise even more than the Laplacian ($\sqrt{20}=4.47$). So it is used with a small gain, usually on an image that was smoothed first.

In [5]:
# Check for 3(a): filter pure noise with each kernel and compare the measured std with the prediction.
kernels = {
    "box":      np.ones((3, 3)) / 9,
    "Gaussian": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
    "Sobel":    np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]),
    "sharpen":  np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}
sigma_eta = 1.0
noise = rng.normal(0, sigma_eta, (1024, 1024))

expected = {"box": 1 / 9, "Gaussian": 36 / 256, "Sobel": 12, "sharpen": 29}
for name, w in kernels.items():
    s2 = (w ** 2).sum()
    measured = ndimage.convolve(noise, w.astype(float), mode="wrap").std()
    print(f"{name:9s} sum w^2 = {s2:.4f}  predicted std {np.sqrt(s2):.3f}  measured {measured:.3f}")
    assert np.isclose(s2, expected[name]) and abs(measured / np.sqrt(s2) - 1) < 0.02
print("five-point Laplacian noise gain sqrt(20) =", np.sqrt(20).round(2))

box       sum w^2 = 0.1111  predicted std 0.333  measured 0.334
Gaussian  sum w^2 = 0.1406  predicted std 0.375  measured 0.376
Sobel     sum w^2 = 12.0000  predicted std 3.464  measured 3.471
sharpen   sum w^2 = 29.0000  predicted std 5.385  measured 5.388
five-point Laplacian noise gain sqrt(20) = 4.47


**(b)** Prove that among *all* $3\times3$ kernels whose weights sum to $1$, the box kernel minimises $\sum_k w_k^{2}$. (One line of Cauchy-Schwarz.) **[5]**

**Solution 3(b).**

By Cauchy-Schwarz, $1=\big(\sum_{k=1}^{9} w_k\cdot 1\big)^{2}\le 9\sum_k w_k^{2}$. So $\sum_k w_k^{2}\ge 1/9$, with equality only when all $w_k$ are equal, i.e. the box.

**(c)** Your answers to (a) and (b) together say that the box filter is a strictly better denoiser, per pixel of support, than the $3\times3$ Gaussian. Yet the Gaussian is mostly the preferred choice. Give one distinct reason why. **[5]**

**Solution 3(c).**

Any one reason gets the full 5 marks.

1. *Frequency response.* The box is a rectangle, so its Fourier transform is a $\operatorname{sinc}$. The sinc ripples and changes sign, so some high frequencies pass through (some with flipped sign). This causes ringing and aliasing. A Gaussian transforms to a Gaussian: positive and smoothly decaying, with no ripples.
2. *Same blur in all directions.* The box weights a corner neighbour as much as a side neighbour, so it blurs more along the diagonals. The Gaussian blurs equally in all directions.
3. *Cascading.* Two Gaussians make one Gaussian, $G_{\sigma_1}\ast G_{\sigma_2}=G_{\sqrt{\sigma_1^2+\sigma_2^2}}$, which gives a clean idea of scale. Two boxes do not make a box.
4. *Fair comparison.* (b) compares kernels of the same size. In practice we care about noise removed for a given loss of detail, and the box destroys fine detail more abruptly.

**(d)** An alternative to spatial smoothing is to average $L$ frames of a static scene. How many frames are needed to match the noise reduction of one $3\times3$ box filter, and what does frame averaging give you that the box filter cannot? **[5]**

**Solution 3(d).**

$\sigma_\eta^2/L=\sigma_\eta^2/9$, so $L=9$ frames. Frame averaging removes noise **without blurring**: it averages the same pixel over time, not different pixels in space, so edges stay sharp. The box filter pays for its noise reduction with lost resolution. The cost of frame averaging is that the scene and the camera must not move.

---
## Question 4. Zero-crossings  [25 marks]

**(a)** Show that the stencil $[1\;\;{-}2\;\;1]$ annihilates every affine ramp $f(x)=ax+b$, and that the five-point $\nabla^{2}$ annihilates every plane $f(x,y)=ax+by+c$. What does this tell you about the response of $\nabla^{2}$ inside a smoothly shaded region, and why is that desirable? **[15]**

**Solution 4(a).**

1D: $f(x{+}1)-2f(x)+f(x{-}1)=(ax+a+b)-2(ax+b)+(ax-a+b)=0$.

2D: the five-point stencil is the $[1\,{-}2\,1]$ stencil along $x$ plus the same stencil along $y$. Along any row or column a plane is an affine ramp, so each part gives $0$, and $\nabla^{2} f=0+0=0$.

Meaning: $\nabla^{2}$ responds only to *bending* of the intensity. A constant region and a region with a constant slope (smooth shading) both give exactly zero. This is good: smooth shading, such as a wall lit from one side, is not an object boundary, so an edge detector should ignore it.

In [6]:
# Check for 4(a).
stencil_1d = np.array([1, -2, 1])
laplacian_5pt = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
x = np.arange(20.0)
X, Y = np.meshgrid(np.arange(20.0), np.arange(20.0))

ramp = 3.7 * x - 2.0
plane = 1.3 * X - 0.8 * Y + 5.0
r1 = np.convolve(ramp, stencil_1d, mode="valid")
r2 = ndimage.convolve(plane, laplacian_5pt.astype(float))[1:-1, 1:-1]
print("max |response| on the ramp:", np.abs(r1).max(), " on the plane:", np.abs(r2).max())
assert np.allclose(r1, 0) and np.allclose(r2, 0)

max |response| on the ramp: 1.4210854715202004e-14  on the plane: 1.7763568394002505e-14


**(b)** A 1D edge is sampled and the second difference takes the values

$$f''(62)=+1.07,\qquad f''(63)=+0.33,\qquad f''(64)=-0.49 .$$

Locate the zero-crossing to sub-pixel accuracy by linear interpolation, showing the arithmetic. The corresponding $|f'|$ values are $14.10,\,14.80,\,14.72$; state the best estimate of the edge location obtainable from $\arg\max|f'|$ alone. **[10]**

**Solution 4(b).**

The sign changes between $x=63$ and $x=64$. Linear interpolation: $t=\dfrac{f''(63)}{f''(63)-f''(64)}=\dfrac{0.33}{0.33+0.49}=\dfrac{0.33}{0.82}=0.402$, so $x^{\ast}\approx63.40$.

$\arg\max|f'|$ can only return a whole pixel: $x=63$ (the largest value, $14.80$). Its error can be up to $0.5$ px. (In the lecture example the true edge was at $63.4$.)

In [7]:
# Check for 4(b).
xs = np.array([62, 63, 64])
f2 = np.array([1.07, 0.33, -0.49])     # second difference
f1 = np.array([14.10, 14.80, 14.72])   # |first difference|

t = f2[1] / (f2[1] - f2[2])
x_zero = xs[1] + t
print(f"t = {t:.3f}, zero-crossing at {x_zero:.2f}, argmax |f'| at {xs[np.argmax(f1)]}")
assert round(x_zero, 2) == 63.40 and xs[np.argmax(f1)] == 63

t = 0.402, zero-crossing at 63.40, argmax |f'| at 63


---
*End of Assignment 1.*